# 🔧 題目 5：求職媒合薪資洞察 — Solution

⚠️ 講師用。學員請用 `pipeline_starter.ipynb`。


## Section 0：環境設定


In [ ]:
# （不需要改）Colab 環境自動設定
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/lu791019/midterm-mvp-template.git /content/repo
    os.chdir('/content/repo/data/raw/topic_5')
    print("✅ Colab：已設定工作目錄 =", os.getcwd())
else:
    print("✅ 本地環境，工作目錄 =", os.getcwd())


In [ ]:
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
# （不需要改）
OPENAI_API_KEY = ""

# 方法 1：Colab Secrets（在左側 🔑 設定 OPENAI_API_KEY）
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

# 方法 2：本地 .env 檔案
if not OPENAI_API_KEY and os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]

print("✅ API Key 已設定" if OPENAI_API_KEY else "⚠️ 無 API Key，使用 fallback（不影響完成度）")


---
## Section 1：Extract


### Step 1-1：讀取 CSV


In [ ]:
df_raw = pd.read_csv("jobs.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
df_raw.head()


### Step 1-2：檢查


In [ ]:
print(df_raw.dtypes)
print("\n", df_raw.isnull().sum())
print("\n", df_raw.describe())


### Step 1-3：自由探索


In [ ]:
print(df_raw.iloc[:, 0].value_counts().head(10))
print(f"\n唯一值: {df_raw.iloc[:, 0].nunique()}")


### Step 1-4：SQLite


In [ ]:
conn = sqlite3.connect("pipeline.db")
df_raw.to_sql("raw_jobs", conn, if_exists="replace", index=False)
print(f"✅ raw_jobs: {pd.read_sql('SELECT COUNT(*) as n FROM raw_jobs', conn)['n'][0]} 筆")


---
## Section 2：Transform


### Step 2-1：從 raw 讀出


In [ ]:
df = pd.read_sql("SELECT * FROM raw_jobs", conn)
before = len(df)
print(f"讀出 {before} 筆")


### Step 2-2 ~ 2-4：清洗


In [ ]:
df = df.dropna(subset=["job_title", "salary_in_usd"])
df["salary_in_usd"] = pd.to_numeric(df["salary_in_usd"], errors="coerce")
df = df[df["salary_in_usd"] > 0]


In [ ]:
print(f"清洗前: {before} → 清洗後: {len(df)}")


### 🏁 檢查點


In [ ]:
assert (df["salary_in_usd"] > 0).all(), "❌ salary 有非正值"
assert df["job_title"].notna().all(), "❌ job_title 有空值"
print("✅ 通過")
print(f"   {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned


In [ ]:
df.to_sql("cleaned_jobs", conn, if_exists="replace", index=False)
print(f"✅ cleaned_jobs")


---
## Section 3：SQL


### Step 3-1：各職位類別平均薪資


In [ ]:
category_salary = pd.read_sql("""
SELECT job_category,
       COUNT(*) as job_count,
       ROUND(AVG(salary_in_usd), 0) as avg_salary,
       ROUND(MIN(salary_in_usd), 0) as min_salary,
       ROUND(MAX(salary_in_usd), 0) as max_salary
FROM cleaned_jobs
GROUP BY job_category
ORDER BY avg_salary DESC
""", conn)
category_salary


### Step 3-2：工作模式 vs 薪資


In [ ]:
setting_salary = pd.read_sql("""
SELECT work_setting, experience_level,
       COUNT(*) as jobs,
       ROUND(AVG(salary_in_usd), 0) as avg_salary
FROM cleaned_jobs
GROUP BY work_setting, experience_level
ORDER BY work_setting, avg_salary DESC
""", conn)
setting_salary


### Step 3-3：視覺化


In [ ]:
import matplotlib.pyplot as plt
category_salary.head(10).plot.barh(x=category_salary.columns[0], y=category_salary.columns[-1], figsize=(10,5))
plt.tight_layout()
plt.show()


### Step 3-5：存結果


In [ ]:
os.makedirs("processed", exist_ok=True)
category_salary.to_csv("processed/category_salary.csv", index=False)
setting_salary.to_csv("processed/setting_salary.csv", index=False)
print("✅ 已存")


---
## Section 4：LLM


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下職稱，回傳 JSON：
{{"field": "工程/分析/科學/管理/其他", "insight": "一句話職涯建議"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1:
            content = content[start:end + 1]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["engineer","developer","architect","devops"]): cat = "工程"
    elif any(w in t for w in ["analyst","analytics","business intelligence","bi"]): cat = "分析"
    elif any(w in t for w in ["scientist","researcher","machine learning","ml","ai"]): cat = "科學"
    elif any(w in t for w in ["manager","director","lead","head","vp"]): cat = "管理"
    else: cat = "其他"
    return {"field": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper")


### Step 4-1：單筆測試


In [ ]:
test = str(df["job_title"].iloc[0])
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"📝 {test[:60]}\n🤖 {result}")


### Step 4-2：批次


In [ ]:
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["job_title"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  {len(results)}/{BATCH_SIZE}")
print(f"✅ {len(results)} 筆")


### Step 4-3：整理 + 寫入


In [ ]:
df_analyzed = df.head(BATCH_SIZE).copy()
first_keys = list(results[0].keys())
for k in first_keys:
    df_analyzed[k] = [r.get(k, "") for r in results]
df_analyzed.rename(columns={k: "llm_insight" for k in first_keys if "insight" in k}, inplace=True)
df_analyzed.to_sql("analyzed_jobs", conn, if_exists="replace", index=False)
print("📊 三表：")
for t in ["raw_jobs", "cleaned_jobs", "analyzed_jobs"]:
    print(f"  {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn)['n'][0]}")


---
## Section 5：驗證


In [ ]:
lineage = pd.read_sql("""
SELECT \'raw_jobs\' as layer, COUNT(*) as rows FROM raw_jobs
UNION ALL SELECT \'cleaned_jobs\', COUNT(*) FROM cleaned_jobs
UNION ALL SELECT \'analyzed_jobs\', COUNT(*) FROM analyzed_jobs
""", conn)
print(lineage.to_string(index=False))


---
## Section 6：報告


In [ ]:
avg_all = df["salary_in_usd"].mean()
salary_lines = "\n".join(
    f'- {r["job_category"]}: ${r["avg_salary"]:,.0f} USD（{r["job_count"]} 筆）'
    for _, r in category_salary.iterrows()
)
report = f"""# 求職媒合薪資分析報告
## 資料概要
- 分析職缺：{len(df)} 筆
- 平均年薪：${avg_all:,.0f} USD
## 各類別薪資
{salary_lines}
## 建議
1. 關注高薪職位的技能要求
2. Remote 工作不一定薪資較低
## Pipeline
CSV → pandas → SQLite → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/pipeline_doc.md", "w") as f: f.write(report)
print("✅ pipeline_doc.md")


---
## Section 7：資料確認與匯出


In [ ]:
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/pipeline_doc.md","報告")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
c = sqlite3.connect("pipeline.db")
for t in ["raw_jobs","cleaned_jobs","analyzed_jobs"]:
    try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', c)['n'][0]}")
    except: print(f"  ❌ {t}")
c.close()


---
## Section 8：FastAPI


In [ ]:
from fastapi import FastAPI, HTTPException
import os

api = FastAPI(title='求職媒合薪資洞察 API')
DB_PATH = "pipeline.db"

def get_conn():
    if not os.path.exists(DB_PATH):
        raise HTTPException(status_code=500, detail="pipeline.db 不存在，請先完成前面 pipeline")
    return sqlite3.connect(DB_PATH)

@api.get("/health")
def health():
    c = get_conn()
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", c)
    c.close()
    return {"status": "ok", "tables": tables["name"].tolist()}

@api.get("/stats")
def stats():
    c = get_conn()
    df_api = pd.read_sql("""SELECT job_category, COUNT(*) as jobs, ROUND(AVG(salary_in_usd),0) as avg_salary FROM cleaned_jobs GROUP BY job_category ORDER BY avg_salary DESC""", c)
    c.close()
    return df_api.to_dict(orient="records")

@api.get("/analyzed")
def analyzed(limit: int = 20):
    c = get_conn()
    safe_limit = min(limit, 100)
    df_api = pd.read_sql(f"""SELECT job_title, field, llm_insight FROM analyzed_jobs LIMIT {limit}""", c)
    c.close()
    return df_api.to_dict(orient="records")

@api.get("/summary")
def summary():
    c = get_conn()
    result = {}
    for table in ['raw_jobs', 'cleaned_jobs', 'analyzed_jobs']:
        try:
            result[table] = int(pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", c)["n"][0])
        except Exception:
            result[table] = 0
    c.close()
    return result

@api.get("/report")
def report():
    path = "output/pipeline_doc.md"
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail="pipeline_doc.md 不存在")
    with open(path) as f:
        return {"report": f.read()}

print("✅ API 定義完成")
print("📡 /health:", health())
print("📡 /stats:", stats()[:3])
print("📡 /analyzed:", analyzed()[:3])
print("📡 /summary:", summary())


In [ ]:
# 啟動 API + 測試
import subprocess, sys, time, requests as req, os

PORT = 8000
if os.path.exists('api.py'):
    proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'api:app',
         '--host', '127.0.0.1', '--port', str(PORT), '--log-level', 'warning'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(20):
        try:
            req.get(f'http://127.0.0.1:{PORT}/health', timeout=1); break
        except: time.sleep(0.5)
    else: print('❌ 啟動失敗')

print('📡 /health:', req.get(f'http://127.0.0.1:{PORT}/health').json())
print('📡 /stats:', req.get(f'http://127.0.0.1:{PORT}/stats').json()[:3])
print('📡 /analyzed:', req.get(f'http://127.0.0.1:{PORT}/analyzed').json()[:3])


> 完整版在 `api.py`。本地啟動請用 Terminal：`uvicorn api:app --reload --port 8000`


---
## Section 9：Dashboard


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

c = sqlite3.connect("pipeline.db")
stats = pd.read_sql("""SELECT job_category, COUNT(*) as jobs, ROUND(AVG(salary_in_usd),0) as avg_salary FROM cleaned_jobs GROUP BY job_category ORDER BY avg_salary DESC""", c)
analyzed_preview = pd.read_sql("SELECT * FROM analyzed_jobs LIMIT 20", c)
c.close()

view = widgets.ToggleButtons(
    options=[("統計", "stats"), ("LLM", "llm")],
    description="資料："
)

def update_dashboard(tab):
    clear_output(wait=True)
    display(view)
    if tab == "stats":
        display(stats.head(20))
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        stats.set_index("job_category")["avg_salary"].sort_values().plot.barh(ax=ax, color="purple")
        ax.set_title("各職類平均薪資")
        plt.tight_layout()
        plt.show()
    else:
        display(analyzed_preview)

widgets.interact(update_dashboard, tab=view)


> 完整版在 `app.py`。本地啟動請用 Terminal：`streamlit run app.py`


---
## Section 10：本地部署指引

完整版已放在同資料夾的 `api.py` 與 `app.py`。在兩個 Terminal 分別執行：

```bash
cd data/raw/topic_5
uvicorn api:app --reload --port 8000
```

```bash
cd data/raw/topic_5
streamlit run app.py
```

API 啟動後可測：
- `http://127.0.0.1:8000/health`
- `http://127.0.0.1:8000/stats`
- `http://127.0.0.1:8000/analyzed`
- `http://127.0.0.1:8000/report`
